# Assignment 1

**Date:** October 28, 2026

**Author:** Tanya Toluay-Coşkun


## Question 1

Compare four methods: zero-shot prompting, four-shot prompting, self-consistency using four
sampled solutions with majority voting, and an initial solution followed by one
reflection/revision call. Finalize prompts using development examples only. Report test accuracy,
total generated tokens, and model calls in one table. Analyze two cases where additional
inference helped or harmed performance, and justify which method provides the best accuracy–
cost tradeoff.

In [ ]:
# Install libraries needed
%pip install -q \
    "pandas==2.2.3" \
    "huggingface-hub>=1.34.0,<3.0" \
    "transformers" \
    "datasets" \
    "accelerate" \
    "sentencepiece"

In [ ]:
import json
import random
import re
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 256
RESULTS_PATH = Path("gsm8k_q1_results.json")

print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Seed:", SEED)

Device: CPU
Seed: 42


In [ ]:
# GSM8K rows are 0-indexed in Python; assignment example 1 is row 0.
dataset = load_dataset("openai/gsm8k", "main")
development = dataset["train"].select(range(0, 10))      # Examples 1–10
demonstrations = dataset["train"].select(range(10, 14))  # Examples 11–14
test_questions = dataset["test"].select(range(0, 30))    # Examples 1–30

assert len(development) == 10
assert len(demonstrations) == 4
assert len(test_questions) == 30

# Don't print test answers or insert them into any model prompt.
print(f"Development: {len(development)} | Four-shot: {len(demonstrations)} | Test: {len(test_questions)}")
print("First development question:", development[0]["question"])

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Development: 10 | Four-shot: 4 | Test: 30
First development question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    model = model.to("cpu")

model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)

DEVICE = next(model.parameters()).device
print("Loaded:", MODEL_ID)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-0.5B-Instruct
Trainable parameters: 0


In [ ]:
NUMBER_PATTERN = r"[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?"
FINAL_PATTERN = re.compile(r"FINAL_ANSWER\s*:\s*\$?\s*(" + NUMBER_PATTERN + r")", re.I)
HASH_PATTERN = re.compile(r"####\s*\$?\s*(" + NUMBER_PATTERN + r")")
ANY_NUMBER_PATTERN = re.compile(NUMBER_PATTERN)


def parse_number(text):
    """Extract a numeric answer, preferring an explicit final-answer marker."""
    text = str(text)
    for pattern in (FINAL_PATTERN, HASH_PATTERN):
        matches = pattern.findall(text)
        if matches:
            return Decimal(matches[-1].replace(",", ""))
    matches = ANY_NUMBER_PATTERN.findall(text)
    return Decimal(matches[-1].replace(",", "")) if matches else None


def reference_answer(answer_text):
    """Use the GSM8K #### answer only in evaluation, never in a model call."""
    matches = HASH_PATTERN.findall(answer_text)
    if not matches:
        raise ValueError("GSM8K reference answer is missing the #### marker")
    return Decimal(matches[-1].replace(",", ""))


def answer_is_correct(predicted, reference):
    return predicted is not None and predicted == reference


# Lightweight parser checks; they use invented strings, not test references.
assert parse_number("FINAL_ANSWER: 1,234") == Decimal(1234)
assert parse_number("The result is -2.5") == Decimal("-2.5")
assert parse_number("No numeric answer") is None
print("Answer parser checks passed.")

Answer parser checks passed.


In [ ]:
SYSTEM_PROMPT = (
    "You solve grade-school math word problems accurately. "
    "Explain the necessary calculations briefly. "
    "End with a separate line in exactly this format: FINAL_ANSWER: <number>. "
    "Do not include units or currency symbols after the number."
)


def zero_shot_messages(question):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Solve this problem:\n{question}"},
    ]


def build_demonstration_text():
    """Training references are permitted in demonstrations, not test references."""
    examples = []
    for row in demonstrations:
        reasoning = row["answer"].split("####")[0].strip()
        final = reference_answer(row["answer"])
        examples.append(
            f"Question: {row['question']}\n"
            f"Worked solution: {reasoning}\n"
            f"FINAL_ANSWER: {final}"
        )
    return "\n\n".join(examples)


FOUR_SHOT_EXAMPLES = build_demonstration_text()


def four_shot_messages(question):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            "Here are four worked examples. Follow their format.\n\n"
            f"{FOUR_SHOT_EXAMPLES}\n\n"
            f"Now solve this new problem:\n{question}"
        )},
    ]


def reflection_messages(question, initial_solution):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": (
            f"Problem:\n{question}\n\n"
            f"Initial solution:\n{initial_solution}\n\n"
            "Check the calculations and reasoning carefully. Correct any mistakes. "
            "Give the revised solution and finish with FINAL_ANSWER: <number>."
        )},
    ]

In [ ]:
def generate_solution(messages, *, sample=False):
    """
    Generate a solution using the frozen Qwen model.

    Parameters
    ----------
    messages : list[dict]
        Chat messages containing the problem and instructions.
    sample : bool
        Whether to use sampling instead of greedy decoding.

    Returns
    -------
    dict
        Generated text, parsed prediction, generated token count,
        and number of model calls.
    """

    # Format the conversation using Qwen's chat template.
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    # Move tokenized inputs to the model's device.
    inputs = inputs.to(DEVICE)

    # Configure generation.
    generation_options = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": sample,
        "pad_token_id": tokenizer.eos_token_id,
    }

    if sample:
        generation_options.update(
            temperature=0.8,
            top_p=0.95
        )

    # Generate without computing gradients.
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            **generation_options
        )

    # Exclude input tokens from the generated output.
    input_length = inputs["input_ids"].shape[-1]
    new_tokens = output[0, input_length:]

    # Decode the generated response.
    generated_text = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "text": generated_text,
        "prediction": parse_number(generated_text),
        "generated_tokens": int(new_tokens.numel()),
        "calls": 1
    }


def serialize_result(result):
    """Make Decimal predictions safe for JSON checkpoint files."""
    if isinstance(result, dict):
        return {k: serialize_result(v) for k, v in result.items()}
    if isinstance(result, list):
        return [serialize_result(v) for v in result]
    if isinstance(result, Decimal):
        return str(result)
    return result

In [ ]:
def run_zero_shot(question):
    return generate_solution(zero_shot_messages(question))


def run_four_shot(question):
    return generate_solution(four_shot_messages(question))


def run_self_consistency(question):
    samples = [generate_solution(zero_shot_messages(question), sample=True)
               for _ in range(4)]
    predictions = [s["prediction"] for s in samples if s["prediction"] is not None]

    if predictions:
        votes = Counter(predictions)
        highest_count = max(votes.values())
        # The input order of predictions preserves the sampling order.
        winner = next(p for p in predictions if votes[p] == highest_count)
    else:
        winner = None

    return {
        "prediction": winner,
        "generated_tokens": sum(s["generated_tokens"] for s in samples),
        "calls": len(samples),
        "samples": samples,
    }


def run_reflection(question):
    initial = generate_solution(zero_shot_messages(question))
    revised = generate_solution(reflection_messages(question, initial["text"]))
    return {
        "prediction": revised["prediction"],
        "generated_tokens": initial["generated_tokens"] + revised["generated_tokens"],
        "calls": initial["calls"] + revised["calls"],
        "initial": initial,
        "revision": revised,
    }


METHODS = {
    "Zero-shot": run_zero_shot,
    "Four-shot": run_four_shot,
    "Self-consistency": run_self_consistency,
    "Reflection": run_reflection,
}

In [ ]:
if RESULTS_PATH.exists():
    with RESULTS_PATH.open("r", encoding="utf-8") as handle:
        results = json.load(handle)
    print("Resuming saved generations from:", RESULTS_PATH)
else:
    results = {}

for index, item in enumerate(test_questions):
    case_id = str(index + 1)  # Assignment-style numbering: 1 to 30
    results.setdefault(case_id, {})

    for method_name, method_fn in METHODS.items():
        if method_name in results[case_id]:
            continue
        print(f"Test {case_id:>2}/30 | {method_name}", flush=True)
        prediction = method_fn(item["question"])
        results[case_id][method_name] = serialize_result(prediction)

        # Write after every completed method so a restart loses minimal work.
        with RESULTS_PATH.open("w", encoding="utf-8") as handle:
            json.dump(results, handle, indent=2, ensure_ascii=False)

print("Finished all model generations. Saved to", RESULTS_PATH)

Test  1/30 | Zero-shot
Test  1/30 | Four-shot
Test  1/30 | Self-consistency
Test  1/30 | Reflection
Test  2/30 | Zero-shot
Test  2/30 | Four-shot
Test  2/30 | Self-consistency
Test  2/30 | Reflection
Test  3/30 | Zero-shot
Test  3/30 | Four-shot
Test  3/30 | Self-consistency
Test  3/30 | Reflection
Test  4/30 | Zero-shot
Test  4/30 | Four-shot
Test  4/30 | Self-consistency
Test  4/30 | Reflection
Test  5/30 | Zero-shot
Test  5/30 | Four-shot
Test  5/30 | Self-consistency
Test  5/30 | Reflection
Test  6/30 | Zero-shot
Test  6/30 | Four-shot
Test  6/30 | Self-consistency
Test  6/30 | Reflection
Test  7/30 | Zero-shot
Test  7/30 | Four-shot
Test  7/30 | Self-consistency
Test  7/30 | Reflection
Test  8/30 | Zero-shot
Test  8/30 | Four-shot
Test  8/30 | Self-consistency
Test  8/30 | Reflection
Test  9/30 | Zero-shot
Test  9/30 | Four-shot
Test  9/30 | Self-consistency
Test  9/30 | Reflection
Test 10/30 | Zero-shot
Test 10/30 | Four-shot
Test 10/30 | Self-consistency
Test 10/30 | Reflection


In [ ]:
records = []
for index, item in enumerate(test_questions):
    case_id = str(index + 1)
    expected = reference_answer(item["answer"])
    for name in METHODS:
        output = results[case_id][name]
        predicted = (Decimal(output["prediction"])
                     if output["prediction"] is not None else None)
        records.append({
            "test_id": index + 1,
            "method": name,
            "prediction": str(predicted) if predicted is not None else None,
            "correct": answer_is_correct(predicted, expected),
            "generated_tokens": output["generated_tokens"],
            "model_calls": output["calls"],
        })

per_question = pd.DataFrame(records)
summary = (per_question.groupby("method", sort=False)
           .agg(correct_answers=("correct", "sum"),
                total_generated_tokens=("generated_tokens", "sum"),
                total_model_calls=("model_calls", "sum"))
           .reset_index())
summary.insert(2, "accuracy_percent", (100 * summary["correct_answers"] / 30).round(2))
summary["avg_tokens_per_question"] = (summary["total_generated_tokens"] / 30).round(1)

assert (summary["total_model_calls"].tolist() == [30, 30, 120, 60])
display(summary)
summary.to_csv("gsm8k_q1_summary.csv", index=False)
per_question.to_csv("gsm8k_q1_per_question.csv", index=False)

,method,correct_answers,accuracy_percent,total_generated_tokens,total_model_calls,avg_tokens_per_question
0,Zero-shot,1,3.33,365,30,12.2
1,Four-shot,4,13.33,7030,30,234.3
2,Self-consistency,1,3.33,2826,120,94.2
3,Reflection,6,20.00,7362,60,245.4


In [ ]:
wide = per_question.pivot(index="test_id", columns="method", values="correct")
comparisons = []
for strategy in ["Self-consistency", "Reflection"]:
    for test_id, row in wide.iterrows():
        if bool(row[strategy]) and not bool(row["Zero-shot"]):
            comparisons.append(("helped", test_id, strategy))
        elif not bool(row[strategy]) and bool(row["Zero-shot"]):
            comparisons.append(("harmed", test_id, strategy))

# Prefer one helpful and one harmful case when available; otherwise pick two distinct cases.
chosen = []
for impact in ["helped", "harmed"]:
    candidate = next((c for c in comparisons
                      if c[0] == impact and c[1] not in [x[1] for x in chosen]), None)
    if candidate:
        chosen.append(candidate)
for candidate in comparisons:
    if len(chosen) >= 2:
        break
    if candidate[1] not in [x[1] for x in chosen]:
        chosen.append(candidate)

if not chosen:
    print("No correct/incorrect changes relative to zero-shot were observed.")
    print("Discuss two cases with changed numerical predictions, if available, "
          "and explicitly state that accuracy did not change.")

for impact, test_id, strategy in chosen:
    question = test_questions[test_id - 1]["question"]
    print("\n" + "=" * 75)
    print(f"TEST {test_id}: extra inference {impact} ({strategy})")
    print("Question:", question)
    print("Ground truth (analysis only):", reference_answer(test_questions[test_id - 1]["answer"]))
    for name in ["Zero-shot", strategy]:
        outcome = results[str(test_id)][name]
        print(f"\n{name}: predicted {outcome['prediction']}")
        if name == "Zero-shot":
            print("Solution:", outcome["text"])
        elif name == "Reflection":
            print("First attempt:", outcome["initial"]["text"])
            print("Revision:", outcome["revision"]["text"])
        else:
            for j, sample in enumerate(outcome["samples"], 1):
                print(f"Sample {j}: {sample['prediction']} | {sample['text'][-200:]}")

print("\nCases found:", len(chosen), "| Available performance changes:", len(comparisons))


TEST 2: extra inference helped (Reflection)
Question: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
Ground truth (analysis only): 3

Zero-shot: predicted 4
Solution: FINAL_ANSWER: 4.

Reflection: predicted 3
First attempt: FINAL_ANSWER: 4.
Revision: To determine the total number of bolts needed for the robe, we need to calculate the amount of each type of fiber required and then sum them up.

1. **Blue Fiber**: The problem states that it takes 2 bolts of blue fiber.
   \[
   \text{Blue Fiber} = 2 \text{ bolts}
   \]

2. **White Fiber**: It is given that the amount of white fiber is half the amount of blue fiber.
   \[
   \text{White Fiber} = \frac{1}{2} \times \text{Blue Fiber} = \frac{1}{2} \times 2 \text{ bolts} = 1 \text{ bolt}
   \]

3. **Total Bolts**: To find the total number of bolts needed, we add the number of blue and white fibers together.
   \[
   \text{Total Bolts} = \text{Blue Fiber} + \text{White Fiber} = 2 \te

In [ ]:
ranked = summary.sort_values(
    ["accuracy_percent", "total_generated_tokens", "total_model_calls"],
    ascending=[False, True, True]
).reset_index(drop=True)

winner = ranked.iloc[0]
print("Best observed accuracy-cost choice under the stated rule:", winner["method"])
print(f"Accuracy: {winner['accuracy_percent']:.2f}%")
print("Total generated tokens:", int(winner["total_generated_tokens"]))
print("Total model calls:", int(winner["total_model_calls"]))
print("\nReport the numerical differences and discuss the tradeoff rather than")
print("assuming that more inference necessarily improves a 0.5B model.")

display(ranked)

Best observed accuracy-cost choice under the stated rule: Reflection
Accuracy: 20.00%
Total generated tokens: 7362
Total model calls: 60

Report the numerical differences and discuss the tradeoff rather than
assuming that more inference necessarily improves a 0.5B model.


,method,correct_answers,accuracy_percent,total_generated_tokens,total_model_calls,avg_tokens_per_question
0,Reflection,6,20.00,7362,60,245.4
1,Four-shot,4,13.33,7030,30,234.3
2,Zero-shot,1,3.33,365,30,12.2
3,Self-consistency,1,3.33,2826,120,94.2


In [ ]:
import re
from collections import Counter

print("=" * 65)
print("QUESTION 1: DIAGNOSTIC CHECK")
print("=" * 65)

methods = ["Zero-shot", "Four-shot", "Self-consistency", "Reflection"]

for method in methods:
    outputs = [results[str(i)][method] for i in range(1, 31)]

    if method == "Reflection":
        responses = [o["revision"] for o in outputs]
    elif method == "Self-consistency":
        responses = [s for o in outputs for s in o["samples"]]
    else:
        responses = outputs

    missing_format = [
        r for r in responses
        if not re.search(r"FINAL_ANSWER\s*:", r["text"], re.I)
    ]

    no_prediction = [
        r for r in responses if r["prediction"] is None
    ]

    near_limit = [
        r for r in responses
        if r["generated_tokens"] >= MAX_NEW_TOKENS
    ]

    lengths = [r["generated_tokens"] for r in responses]

    print(f"\n{method}")
    print(f"  Responses:             {len(responses)}")
    print(f"  Missing FINAL_ANSWER:  {len(missing_format)}")
    print(f"  Missing predictions:   {len(no_prediction)}")
    print(f"  Hit generation limit:  {len(near_limit)}")
    print(f"  Average output tokens: {sum(lengths)/len(lengths):.1f}")

# Inspect zero-shot responses
print("\n" + "=" * 65)
print("ZERO-SHOT OUTPUT EXAMPLES")
print("=" * 65)

for i in range(1, 6):
    r = results[str(i)]["Zero-shot"]
    print(f"\nTest {i}")
    print("Prediction:", r["prediction"])
    print("Tokens:", r["generated_tokens"])
    print("Output:", repr(r["text"][:500]))

# Inspect reflection failure
print("\n" + "=" * 65)
print("REFLECTION: TEST 8")
print("=" * 65)

r = results["8"]["Reflection"]

print("Initial answer:", r["initial"]["prediction"])
print("Revised answer:", r["revision"]["prediction"])
print("Revision tokens:", r["revision"]["generated_tokens"])
print("Revision text:")
print(r["revision"]["text"][-1200:])

# Check self-consistency voting
print("\n" + "=" * 65)
print("SELF-CONSISTENCY: FIRST 5 QUESTIONS")
print("=" * 65)

for i in range(1, 6):
    r = results[str(i)]["Self-consistency"]
    predictions = [s["prediction"] for s in r["samples"]]

    print(
        f"Test {i}: "
        f"Samples={predictions}, "
        f"Votes={dict(Counter(str(p) for p in predictions))}, "
        f"Selected={r['prediction']}"
    )

QUESTION 1: DIAGNOSTIC CHECK

Zero-shot
  Responses:             30
  Missing FINAL_ANSWER:  0
  Missing predictions:   0
  Hit generation limit:  0
  Average output tokens: 12.2

Four-shot
  Responses:             30
  Missing FINAL_ANSWER:  30
  Missing predictions:   0
  Hit generation limit:  17
  Average output tokens: 234.3

Self-consistency
  Responses:             120
  Missing FINAL_ANSWER:  0
  Missing predictions:   0
  Hit generation limit:  2
  Average output tokens: 23.6

Reflection
  Responses:             30
  Missing FINAL_ANSWER:  22
  Missing predictions:   0
  Hit generation limit:  14
  Average output tokens: 233.2

ZERO-SHOT OUTPUT EXAMPLES

Test 1
Prediction: 384
Tokens: 10
Output: 'FINAL_ANSWER: 384.'

Test 2
Prediction: 4
Tokens: 8
Output: 'FINAL_ANSWER: 4.'

Test 3
Prediction: 30000
Tokens: 13
Output: 'FINAL_ANSWER: $30,000.'

Test 4
Prediction: 1800
Tokens: 11
Output: 'FINAL_ANSWER: 1800.'

Test 5
Prediction: 65
Tokens: 9
Output: 'FINAL_ANSWER: 65.'

REFLECTI

### Interpretation of Results

The results show that increasing inference-time computation does not necessarily improve the mathematical reasoning performance of a small language model.

**Reflection achieved the highest accuracy (20.00%)**, followed by four-shot prompting (13.33%). Zero-shot prompting and self-consistency both achieved 3.33%. Although reflection produced the best results, its accuracy remained relatively low, indicating limitations in the mathematical reasoning capabilities of Qwen2.5-0.5B-Instruct.

**Impact of prompting strategies:** Four-shot prompting outperformed zero-shot prompting by 10 percentage points, suggesting that providing worked examples helped the model approach mathematical word problems. However, four-shot responses were substantially longer, averaging 234.3 generated tokens compared with 12.2 for zero-shot prompting.

**Impact of self-consistency:** Self-consistency did not improve accuracy compared with zero-shot prompting, despite requiring four times as many model calls. For each question, four independently sampled responses were generated, and the most frequently occurring numerical answer was selected. When multiple answers received the same highest vote count, the answer appearing first among the sampled responses was selected as a tie-breaking rule. Therefore, the implementation used plurality voting rather than requiring a strict majority. The sampled answers were often inconsistent. For example, all four candidates for Test 1 produced different answers, while Test 4 resulted in a tie. These findings suggest that sampling multiple solutions does not necessarily improve performance when the underlying model produces inconsistent reasoning.

**Impact of reflection:** Reflection improved the accuracy from 3.33% in zero-shot prompting to 20.00%, demonstrating that an additional revision step can sometimes correct initial mistakes. In Test 2, reflection successfully corrected the initial prediction from 4 to 3 by explicitly calculating the required quantities. However, reflection was not consistently beneficial. In Test 8, the model initially predicted the correct answer of 160 but introduced incorrect assumptions during revision, ultimately producing an incomplete response from which 40 was extracted.

**Generation limits and evaluation limitations:** A diagnostic analysis showed that 17 of 30 four-shot responses and 14 of 30 reflection responses reached the maximum generation limit of 256 tokens. Additionally, all 30 four-shot responses and 22 reflection responses lacked the requested FINAL_ANSWER marker. When an explicit final answer was unavailable, the extraction procedure used the last numerical value in the generated response as the prediction. However, this value could represent an intermediate calculation rather than the model's intended final answer, potentially leading to incorrect classifications of both successful and unsuccessful responses. Therefore, the reported accuracy reflects performance under the specific prompting, token-budget, and answer-extraction settings used in this experiment. These limitations should be considered when interpreting differences between methods, as the results do not necessarily represent the models' full reasoning capabilities.

**Accuracy-cost tradeoff:** Reflection achieved the highest observed accuracy with 7,362 generated tokens and 60 model calls. Four-shot prompting required a similar number of generated tokens (7,030) but only 30 model calls, while achieving 13.33% accuracy. Thus, reflection was the preferred method when prioritizing accuracy, whereas four-shot prompting represented a lower-call alternative. Self-consistency was inefficient in this experiment because it required 120 model calls without improving accuracy over zero-shot prompting.

Overall, the experiment suggests that additional inference can help when it encourages meaningful problem decomposition and error correction, but it can also introduce incorrect reasoning, produce inconsistent solutions, or exhaust the generation budget. Therefore, the effectiveness of inference-time strategies should be evaluated together with their computational cost and implementation constraints.



---



## Question 2

Build an agent that translates each question into arithmetic expressions, calls a restricted Python
calculator, observes its results or errors, and produces a final answer. Allow a maximum of three
calculator calls per question; do not use unrestricted eval or exec. Compare accuracy and token
cost with your strongest method from Question 1. Analyze two failures, distinguishing incorrect
problem formulation from arithmetic errors, and explain why successful tool execution does not
guarantee correctness.

In [ ]:
import ast
import json
import math
import operator
import random
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_NEW_TOKENS = 256  # Same cap as the initial Question 1 run
MAX_CALCULATOR_CALLS = 3
Q2_PATH = Path("gsm8k_q2_results.json")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Seed:", SEED)

dataset = load_dataset("openai/gsm8k", "main")
development = dataset["train"].select(range(0, 10))
demonstrations = dataset["train"].select(range(10, 14))  # Reserved for Q1.
test_questions = dataset["test"].select(range(0, 30))
assert (len(development), len(demonstrations), len(test_questions)) == (10, 4, 30)
print("Data ready: 10 development, 4 demonstrations, 30 test questions.")


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
print("Model loaded; parameters frozen.")


Device: cpu
Seed: 42
Data ready: 10 development, 4 demonstrations, 30 test questions.


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded; parameters frozen.


In [ ]:
NUMBER_PATTERN = r"[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?"
FINAL_PATTERN = re.compile(r"FINAL_ANSWER\s*:\s*\$?\s*(" + NUMBER_PATTERN + r")", re.I)
REFERENCE_PATTERN = re.compile(r"####\s*\$?\s*(" + NUMBER_PATTERN + r")")


def parse_final(text):
    """Only an explicit final answer is eligible for grading."""
    matches = FINAL_PATTERN.findall(str(text))
    return Decimal(matches[-1].replace(",", "")) if matches else None


def reference_answer(text):
    """For offline scoring only; never pass to the agent or calculator."""
    matches = REFERENCE_PATTERN.findall(text)
    if not matches:
        raise ValueError("Missing GSM8K reference marker")
    return Decimal(matches[-1].replace(",", ""))


assert parse_final("FINAL_ANSWER: $1,234.5") == Decimal("1234.5")
assert parse_final("Working: 4 + 3 = 7") is None
print("Final-answer parser checks passed.")


Final-answer parser checks passed.


In [ ]:
import ast
import math
import operator

# Allowed arithmetic operations
BINARY_OPERATORS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}

UNARY_OPERATORS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}


def restricted_calculator(expression):
    """
    Safely evaluate arithmetic expressions using Python's AST.

    Supported operations:
        Addition, subtraction, multiplication, division,
        exponentiation, and unary positive/negative.

    Disallowed:
        Variables, functions, imports, file access, and arbitrary code.

    Parameters
    ----------
    expression : str
        Arithmetic expression to evaluate.

    Returns
    -------
    int or float
        Calculated numerical result.

    Raises
    ------
    ValueError
        If the expression is invalid or unsafe.
    """

    expression = expression.strip()

    # Validate expression length
    if not expression or len(expression) > 200:
        raise ValueError("Expression must contain 1–200 characters")

    # Parse expression safely
    try:
        tree = ast.parse(expression, mode="eval")
    except SyntaxError as exc:
        raise ValueError(f"Invalid arithmetic expression: {exc}") from exc

    def visit(node, depth=0):
        """Recursively evaluate permitted AST nodes."""

        if depth > 25:
            raise ValueError("Expression is too deeply nested")

        if isinstance(node, ast.Expression):
            return visit(node.body, depth + 1)

        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            value = float(node.value)

        elif isinstance(node, ast.UnaryOp) and type(node.op) in UNARY_OPERATORS:
            operand = visit(node.operand, depth + 1)
            value = UNARY_OPERATORS[type(node.op)](operand)

        elif isinstance(node, ast.BinOp) and type(node.op) in BINARY_OPERATORS:
            left = visit(node.left, depth + 1)
            right = visit(node.right, depth + 1)

            if isinstance(node.op, ast.Pow) and abs(right) > 12:
                raise ValueError("Exponent too large")

            value = BINARY_OPERATORS[type(node.op)](left, right)

        else:
            raise ValueError("Only numeric arithmetic is permitted")

        # Prevent excessively large or nonfinite results
        if not math.isfinite(value) or abs(value) > 1e15:
            raise ValueError("Result is not finite or exceeds calculator limits")

        return value

    try:
        result = visit(tree)
    except (ZeroDivisionError, OverflowError, TypeError) as exc:
        raise ValueError(f"Invalid arithmetic: {exc}") from exc

    return int(result) if result.is_integer() else result


# Test valid arithmetic
assert restricted_calculator("(2 + 3) * 4") == 20
assert restricted_calculator("2 ** 3") == 8
assert restricted_calculator("48 / 2") == 24

# Test unsafe expressions
unsafe_expressions = [
    '__import__("os")',
    "open('file')",
    "sum([1,2])",
    "x + 1",
    "2**999",
]

for expression in unsafe_expressions:
    try:
        restricted_calculator(expression)
    except ValueError:
        continue

    raise AssertionError(f"Unsafe expression was accepted: {expression}")

print("Calculator arithmetic and safety tests passed.")

Calculator arithmetic and safety tests passed.


In [ ]:
AGENT_SYSTEM = (
    "You are a grade-school arithmetic agent. "
    "Translate the word problem into numeric arithmetic. "
    "You have a calculator for arithmetic expressions. "
    "At each turn output exactly ONE line starting with CALC: followed by a Python-style "
    "numeric expression using only + - * / ** and parentheses, OR output "
    "FINAL_ANSWER: followed by a number. "
    "You may request at most three calculator calls. "
    "After a calculator observation, use the result to decide whether another "
    "calculation is necessary; otherwise give FINAL_ANSWER. "
    "Do not write explanations, code, variables, functions, or multiple commands. "
    "For example, for three packs of four pencils, send CALC: 3 * 4; "
    "after observing 12, send FINAL_ANSWER: 12."
)
CALC_LINE = re.compile(r"^\s*CALC\s*:\s*(.+?)\s*$", re.I | re.M)


def model_reply(messages):
    """One frozen-model call. Return response text and generated-token cost."""
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    ).to(DEVICE)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0, inputs["input_ids"].shape[-1]:]
    return {
        "text": tokenizer.decode(new_tokens, skip_special_tokens=True).strip(),
        "generated_tokens": int(new_tokens.numel()),
    }


In [ ]:
def run_calculator_agent(question):
    """Execute at most 3 restricted tool calls and collect a complete trace."""
    messages = [
        {"role": "system", "content": AGENT_SYSTEM},
        {"role": "user", "content": f"Solve using the calculator when needed:\n{question}"},
    ]
    trace = []
    generated_tokens = 0
    model_calls = 0
    calculator_calls = 0
    final_text = ""
    stop_reason = ""

    for step in range(MAX_CALCULATOR_CALLS + 1):
        if step == MAX_CALCULATOR_CALLS:
            messages.append({
                "role": "user",
                "content": "No calculator calls remain. Give only FINAL_ANSWER: <number>."
            })
        response = model_reply(messages)
        model_calls += 1
        generated_tokens += response["generated_tokens"]
        text = response["text"]
        messages.append({"role": "assistant", "content": text})
        trace.append({"type": "model", "text": text, "tokens": response["generated_tokens"]})

        # Only accept a final answer when the message is itself a final-answer line.
        if re.fullmatch(r"\s*FINAL_ANSWER\s*:\s*\$?\s*" + NUMBER_PATTERN + r"\.?\s*", text, re.I):
            final_text = text
            stop_reason = "final_answer"
            break

        # After the third tool call, there is no further calculator action.
        if step == MAX_CALCULATOR_CALLS:
            final_text = text
            stop_reason = "final_format_missing"
            break

        match = CALC_LINE.fullmatch(text)
        if not match:
            messages.append({
                "role": "user",
                "content": "Invalid format. Reply using exactly CALC: <expression> or FINAL_ANSWER: <number>."
            })
            trace.append({"type": "format_error", "text": "Unrecognized action format"})
            continue

        expression = match.group(1)
        calculator_calls += 1
        try:
            value = restricted_calculator(expression)
            observation = str(value)
            success = True
        except (ValueError, ArithmeticError) as exc:
            observation = f"ERROR: {exc}"
            success = False
        trace.append({
            "type": "calculator", "expression": expression,
            "observation": observation, "success": success
        })
        messages.append({
            "role": "user",
            "content": f"Calculator observation: {observation}\n"
                       "What is your next single action? Reply CALC: <expression> "
                       "or FINAL_ANSWER: <number>."
        })

    prediction = parse_final(final_text) if stop_reason == "final_answer" else None
    assert calculator_calls <= MAX_CALCULATOR_CALLS
    assert model_calls <= MAX_CALCULATOR_CALLS + 1
    return {
        "prediction": str(prediction) if prediction is not None else None,
        "generated_tokens": generated_tokens,
        "model_calls": model_calls,
        "calculator_calls": calculator_calls,
        "stop_reason": stop_reason,
        "trace": trace,
    }


In [ ]:
for index in (0, 1):
    item = development[index]
    outcome = run_calculator_agent(item["question"])
    expected = reference_answer(item["answer"])  # Development only.
    print(f"Development {index + 1}: predicted={outcome['prediction']} "
          f"reference={expected} calls={outcome['calculator_calls']}")
    for event in outcome["trace"]:
        print(" ", event)
    print()


Development 1: predicted=5760 reference=72 calls=1
  {'type': 'model', 'text': 'CALC: 48/2; CALC: 48-48/2; CALC: 48+48/2; CALC: 48*48/2; CALC: 48**48/2; CALC: 48**(48+48)/2; FINAL_ANSWER: 5760', 'tokens': 77}
  {'type': 'calculator', 'expression': '48/2; CALC: 48-48/2; CALC: 48+48/2; CALC: 48*48/2; CALC: 48**48/2; CALC: 48**(48+48)/2; FINAL_ANSWER: 5760', 'observation': 'ERROR: Invalid arithmetic expression: invalid syntax (<unknown>, line 1)', 'success': False}
  {'type': 'model', 'text': 'FINAL_ANSWER: 5760', 'tokens': 10}

Development 2: predicted=10.0 reference=10 calls=2
  {'type': 'model', 'text': 'CALC: 12 * 50 / 60', 'tokens': 15}
  {'type': 'calculator', 'expression': '12 * 50 / 60', 'observation': '10', 'success': True}
  {'type': 'model', 'text': 'CALC: 12 * 50 / 60', 'tokens': 15}
  {'type': 'calculator', 'expression': '12 * 50 / 60', 'observation': '10', 'success': True}
  {'type': 'model', 'text': 'FINAL_ANSWER: 10.0', 'tokens': 10}



In [ ]:
if Q2_PATH.exists():
    q2_results = json.loads(Q2_PATH.read_text(encoding="utf-8"))
    print("Resuming", Q2_PATH)
else:
    q2_results = {}

for index, item in enumerate(test_questions, start=1):
    case_id = str(index)
    if case_id in q2_results:
        continue
    print(f"Question 2 agent | Test {index:2d}/30", flush=True)
    q2_results[case_id] = run_calculator_agent(item["question"])
    Q2_PATH.write_text(json.dumps(q2_results, indent=2, ensure_ascii=False), encoding="utf-8")

assert len(q2_results) >= 30
print("Saved", Q2_PATH)


Question 2 agent | Test  1/30
Question 2 agent | Test  2/30
Question 2 agent | Test  3/30
Question 2 agent | Test  4/30
Question 2 agent | Test  5/30
Question 2 agent | Test  6/30
Question 2 agent | Test  7/30
Question 2 agent | Test  8/30
Question 2 agent | Test  9/30
Question 2 agent | Test 10/30
Question 2 agent | Test 11/30
Question 2 agent | Test 12/30
Question 2 agent | Test 13/30
Question 2 agent | Test 14/30
Question 2 agent | Test 15/30
Question 2 agent | Test 16/30
Question 2 agent | Test 17/30
Question 2 agent | Test 18/30
Question 2 agent | Test 19/30
Question 2 agent | Test 20/30
Question 2 agent | Test 21/30
Question 2 agent | Test 22/30
Question 2 agent | Test 23/30
Question 2 agent | Test 24/30
Question 2 agent | Test 25/30
Question 2 agent | Test 26/30
Question 2 agent | Test 27/30
Question 2 agent | Test 28/30
Question 2 agent | Test 29/30
Question 2 agent | Test 30/30
Saved gsm8k_q2_results.json


In [32]:
agent_rows = []
for index, item in enumerate(test_questions, start=1):
    outcome = q2_results[str(index)]
    expected = reference_answer(item["answer"])
    predicted = Decimal(outcome["prediction"]) if outcome["prediction"] is not None else None
    agent_rows.append({
        "test_id": index,
        "prediction": str(predicted) if predicted is not None else None,
        "correct": predicted is not None and predicted == expected,
        "generated_tokens": outcome["generated_tokens"],
        "model_calls": outcome["model_calls"],
        "calculator_calls": outcome["calculator_calls"],
        "stop_reason": outcome["stop_reason"],
    })

agent_df = pd.DataFrame(agent_rows)
agent_metrics = {
    "method": "Calculator agent",
    "correct_answers": int(agent_df["correct"].sum()),
    "accuracy_percent": round(100 * agent_df["correct"].mean(), 2),
    "total_generated_tokens": int(agent_df["generated_tokens"].sum()),
    "total_model_calls": int(agent_df["model_calls"].sum()),
    "total_calculator_calls": int(agent_df["calculator_calls"].sum()),
}
display(pd.DataFrame([agent_metrics]))
agent_df.to_csv("gsm8k_q2_per_question.csv", index=False)
print("Stop reasons:", agent_df["stop_reason"].value_counts().to_dict())


,method,correct_answers,accuracy_percent,total_generated_tokens,total_model_calls,total_calculator_calls
0,Calculator agent,3,10.0,1722,79,37


Stop reasons: {'final_answer': 30}


In [33]:
reflection_baseline = {
    "method": "Reflection (Q1 original)",
    "correct_answers": 6,
    "accuracy_percent": 20.00,
    "total_generated_tokens": 7362,
    "total_model_calls": 60,
    "total_calculator_calls": 0,
}

q1_path = Path("gsm8k_q1_results.json")
if q1_path.exists():
    q1_results = json.loads(q1_path.read_text(encoding="utf-8"))
    if all(str(i) in q1_results and "Reflection" in q1_results[str(i)] for i in range(1, 31)):
        reflection_correct = 0
        for i, item in enumerate(test_questions, start=1):
            p = q1_results[str(i)]["Reflection"]["prediction"]
            reflection_correct += p is not None and Decimal(str(p)) == reference_answer(item["answer"])
        reflection_baseline.update({
            "correct_answers": int(reflection_correct),
            "accuracy_percent": round(100 * reflection_correct / 30, 2),
            "total_generated_tokens": sum(q1_results[str(i)]["Reflection"]["generated_tokens"] for i in range(1, 31)),
            "total_model_calls": sum(q1_results[str(i)]["Reflection"]["calls"] for i in range(1, 31)),
        })
        print("Reflection baseline read from Q1 checkpoint.")
else:
    print("Using original Reflection results supplied from Question 1.")

comparison = pd.DataFrame([reflection_baseline, agent_metrics])
display(comparison)
print("Agent minus Reflection accuracy (percentage points):",
      round(agent_metrics["accuracy_percent"] - reflection_baseline["accuracy_percent"], 2))
print("Agent minus Reflection generated tokens:",
      agent_metrics["total_generated_tokens"] - reflection_baseline["total_generated_tokens"])
print("Agent minus Reflection model calls:",
      agent_metrics["total_model_calls"] - reflection_baseline["total_model_calls"])
comparison.to_csv("gsm8k_q2_vs_reflection.csv", index=False)


Reflection baseline read from Q1 checkpoint.


,method,correct_answers,accuracy_percent,total_generated_tokens,total_model_calls,total_calculator_calls
0,Reflection (Q1 original),6,20.0,7362,60,0
1,Calculator agent,3,10.0,1722,79,37


Agent minus Reflection accuracy (percentage points): -10.0
Agent minus Reflection generated tokens: -5640
Agent minus Reflection model calls: 19


In [34]:
failed_ids = agent_df.loc[~agent_df["correct"], "test_id"].tolist()
print("Number of failed questions:", len(failed_ids))
for case_id in failed_ids[:5]:
    item = test_questions[case_id - 1]
    outcome = q2_results[str(case_id)]
    print("\n" + "=" * 65)
    print(f"TEST {case_id}: {item['question']}")
    print("Reference (analysis only):", reference_answer(item["answer"]))
    print("Prediction:", outcome["prediction"], "| Stop:", outcome["stop_reason"])
    for event in outcome["trace"]:
        print(event)

print("\nSelect two instructive failed cases from the traces above (or change [:5] to show more).")


Number of failed questions: 27

TEST 1: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
Reference (analysis only): 18
Prediction: 1000 | Stop: final_answer
{'type': 'model', 'text': 'CALC: (16 - 3) * 16 / 7 * 2', 'tokens': 21}
{'type': 'calculator', 'expression': '(16 - 3) * 16 / 7 * 2', 'observation': '59.42857142857143', 'success': True}
{'type': 'model', 'text': 'FINAL_ANSWER: 1000', 'tokens': 10}

TEST 2: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
Reference (analysis only): 3
Prediction: 2.5 | Stop: final_answer
{'type': 'model', 'text': 'CALC: (2 * 1) + (1/2 * 1)', 'tokens': 19}
{'type': 'calculator', 'expression': '(2 * 1) + (1/2 * 1)', 'observation': '2.5', 'success': True}
{'type': 'mode

### Results and Interpretation

The calculator-based agent correctly answered 3 out of 30 GSM8K test questions, achieving an accuracy of 10%. It generated 1,722 tokens, made 79 model calls, and performed 37 calculator calls.

Compared with Reflection from Question 1, which achieved 20% accuracy using 7,362 generated tokens and 60 model calls, the calculator agent performed worse despite having access to an external arithmetic tool. Although the agent generated 76.6% fewer tokens, it required more model calls (79 versus 60), as well as 37 calculator calls. This shows that lower token consumption does not necessarily mean lower overall computational cost.

**Failure analysis:** In Test 2, the model incorrectly translated the word problem into the expression (2 * 1) + (1/2 * 1), producing 2.5 instead of the correct answer of 3. The calculator evaluated the expression correctly, but the mathematical formulation was wrong. This represents a reasoning error rather than an arithmetic error.

In Test 4, the model generated a malformed calculator command and omitted a multiplication factor from the word problem. The calculator rejected the invalid expression, and the model eventually predicted 180 instead of 540. This example illustrates both incorrect problem formulation and a tool-use error. The calculator itself did not perform an incorrect calculation.

**Conclusion:** Successful calculator execution does not guarantee a correct answer because the calculator only evaluates the expression provided by the model. It cannot determine whether the expression correctly represents the word problem.

The results suggest that the main limitation was not arithmetic computation, but the model's ability to translate natural language into appropriate mathematical expressions and interact reliably with the calculator. Although the calculator agent was more token-efficient than Reflection, it achieved lower accuracy and required more model calls. This suggests that simply providing an external tool is insufficient when the model struggles with problem formulation and tool use.



---



## Question 3

Read this article “Towards autonomous medical artificial intelligence agents” by Ferber et al.”
and identify one potential methodological flaw in its patient simulation or agent evaluation. Cite
the relevant passage or figure, explain how the issue could affect the conclusions, and propose
your suggestion(s).

### Critique of Ferber et al. (2026)

**Identified limitation: The realism of simulated patient interactions**

One potential methodological limitation of Ferber et al. (2026) is the use of simulated patients who may provide more structured and consistent information than patients in real clinical settings.

In the study, the authors developed MIRA, an autonomous medical AI agent, and evaluated its performance using simulated clinical interactions based on retrospective patient records from MIMIC-IV. The simulated patient agent was instructed to respond according to the documented history of present illness (HPI). In Figure 2a, the authors reported 99.4% consistency between responses to semantically equivalent questions. While this shows that the patient agent reliably reproduced the available clinical information, I would argue that consistency alone does not necessarily indicate realistic patient behaviour.

The authors acknowledge this limitation in the Discussion, noting that patient responses generated from discharge summaries may be more structured than real patient speech and may not adequately capture missing information or inconsistencies.

This is important because real-world clinical information is often incomplete or inaccurate. For example, Mazer et al. (2011) found discrepancies in medication histories for 37% of 1,657 emergency department patients. Similarly, a qualitative study on information management in emergency departments identified incorrect, outdated, conflicting, incomplete, and missing information as common challenges encountered by healthcare professionals.

In my view, this could affect the generalizability of the study's findings. MIRA achieved a diagnostic accuracy of 87.8%, compared with 78.1% for board-certified physicians. However, these results were obtained in a controlled simulation environment where patient responses were grounded in documented clinical histories. In practice, physicians must often identify missing information, clarify ambiguous statements, and reconcile conflicting accounts before making clinical decisions.

Although both MIRA and the physicians interacted with the same simulated patients, this does not necessarily mean that the limitations of the simulation affected them equally. Human physicians and AI agents may have different abilities to recognize unreliable information or ask appropriate follow-up questions. Therefore, the relative performance observed in the simulation may differ in real clinical settings.

**Suggested improvement**

I would suggest extending the evaluation by systematically introducing different levels of information quality into the patient simulations. For example, the authors could simulate patients who provide incomplete medical histories, use vague descriptions of symptoms, forget relevant information, or give contradictory answers.

Both MIRA and human physicians could then be evaluated on the same modified cases. In addition to diagnostic accuracy, I would measure whether they recognize inconsistencies, ask appropriate clarification questions, and make safe decisions when information is insufficient. Comparing performance across these conditions would help determine how robust the agent is to the uncertainty commonly encountered in clinical practice.

Overall, I consider the study an important step towards evaluating autonomous medical AI agents in more realistic clinical workflows. However, reliable performance in a controlled simulation does not necessarily translate into equally reliable performance in real clinical environments. Incorporating more realistic patient communication and information-quality challenges would strengthen the evidence for the clinical applicability of such systems.

### References

1. Ferber, D., et al. (2026). Towards autonomous medical artificial intelligence agents. *Nature, 655*, 1282–1291. https://doi.org/10.1038/s41586-026-10675-5

2. Mazer, M., DeRoos, F., Hollander, J. E., McCusker, C., Peacock, N., & Perrone, J. (2011). Medication history taking in emergency department triage is inaccurate and incomplete. *Academic Emergency Medicine, 18*(1), 102–104. https://doi.org/10.1111/j.1553-2712.2010.00959.x

3. *Identification and management of information problems by emergency department staff.* (2015). Available at: https://pmc.ncbi.nlm.nih.gov/articles/PMC4419920/